# 02 — Mean reversion: pesquisa com custos e validação

Sucessor do `01_forex_mean_rev.ipynb`, que **permanece intocado** como referência.

O 01 validou a *lógica de sinal* (RSI-2 de Wilder + streak em zona + alinhamento
SMA21/SMA88) e produziu a lista de parâmetros por ativo. O que ele mede, porém, é
retorno bruto de preço close-a-close: sem custos, sem restrição de uma posição por vez,
sem separação treino/teste. Este notebook parte de onde ele chegou e pergunta se sobra
alguma coisa depois dessas três correções.

A lógica vive agora no pacote `meanrev/`, com testes. O notebook é só a análise.

## Roteiro

| Seção | Pergunta |
|---|---|
| 0 | O motor novo reproduz o notebook 01? |
| 1 | Quanto cada correção custa, uma de cada vez? |
| 2 | Esses ativos são reversíveis à média, antes de otimizar qualquer coisa? |
| 3 | O spread come o edge? Em que horários? |
| 4 | A saída por X candles vale a pena? |
| 5 | O parâmetro escolhido sobrevive fora da amostra? |
| 6 | O resultado se distingue de ruído? |
| 7 | Exportar para o `risk-mgmt`. |


In [1]:
import sys
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append("..")

from meanrev import data, engine, exits, export, indicators, metrics, validation
from meanrev.costs import FREE, CostModel
from meanrev.engine import NOTEBOOK_01, ExecParams, run_backtest
from meanrev.exits import ExitRules
from meanrev.metrics import Sizing, compute_stats, to_money
from meanrev.signals import SignalParams

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_theme(style="whitegrid")


## Universo declarado

Declarar o universo **antes** de procurar é o que torna honesto o orçamento de busca da
seção 6. A lista abaixo é a cesta do notebook 01 restrita ao que já está em cache; mexer
nela depois de ver resultados é o começo do overfitting.


In [ ]:
TIMEFRAME = "M30"
START = dt.datetime(2022, 1, 1)
END = dt.datetime(2026, 9, 18)

# Ativo de trabalho das seções 0-6. Os parâmetros são os que o notebook 01 registrou
# para ele na célula 34 - escolhidos in-sample, que é exatamente o que a seção 5 testa.
ASSET = "AUDCHF"
NB01_STREAK = 1
NB01_EXIT_BARS = 7

UNIVERSE = ["AUDCHF", "AUDJPY", "EURUSD", "GBPUSD", "NAS100"]

df = data.load_rates(ASSET, TIMEFRAME, START, END, allow_download=False)
POINT = data.point_size(df["close"])

print(f"{ASSET} {TIMEFRAME}: {len(df):,} barras de {df['time'].min()} a {df['time'].max()}")
print(f"point = {POINT}")


## 0. Paridade com o notebook 01

Antes de qualquer conclusão nova, o motor tem que reproduzir o antigo. Rodando em
`NOTEBOOK_01` (entrada no fechamento da barra de sinal, saída no fechamento N barras
depois, trades sobrepostos permitidos, custo zero), o total tem que bater com o que o
01 reportava.

Se esta célula não bater, nada abaixo dela vale.


In [ ]:
parity = run_backtest(
    df, ASSET, SignalParams(streak=NB01_STREAK), ExitRules(time_bars=NB01_EXIT_BARS),
    costs=FREE, exec_params=NOTEBOOK_01,
)

print(parity.summary())
print(f"\nRetorno bruto somado: {parity.trades['gross_pct'].sum():+.2f}%")
print("(o teste tests/test_parity_nb01.py trava esse número contra uma transcrição")
print(" literal das células do notebook 01)")


## 1. Cascata: bruto → líquido

Cada degrau liga **uma** correção. A coluna `Δ` é o que aquela correção custou.

1. **Método 01** — o ponto de partida.
2. **Entrada na barra seguinte** — o EA avalia a barra fechada e entra a mercado no tick
   seguinte; o 01 assume execução no fechamento que gerou o sinal.
3. **Uma posição por vez** — as janelas `next_N` do 01 se sobrepõem, somando trades que a
   conta nunca poderia ter tido simultaneamente.
4. **Spread** — a coluna `spread` dos próprios rates, que o 01 nunca usou.
5. **Comissão** e 6. **Swap** — dependem dos valores reais da corretora (ver adiante).


In [ ]:
# Comissão e swap da corretora: PREENCHER. Zero aqui não significa "sem custo", significa
# "ainda não medido" - o cabeçalho de um relatório do Strategy Tester traz os dois.
COMMISSION_PER_LOT = 0.0
SWAP_LONG_PER_LOT = 0.0
SWAP_SHORT_PER_LOT = 0.0

sp = SignalParams(streak=NB01_STREAK)
er = ExitRules(time_bars=NB01_EXIT_BARS)

steps = {}
steps["1. metodo notebook 01"] = run_backtest(df, ASSET, sp, er, FREE, NOTEBOOK_01)
steps["2. + entrada na barra seguinte"] = run_backtest(
    df, ASSET, sp, er, FREE, ExecParams(allow_overlap=True)
)
steps["3. + uma posicao por vez"] = run_backtest(df, ASSET, sp, er, FREE, ExecParams())
steps["4. + spread real"] = run_backtest(df, ASSET, sp, er, CostModel(), ExecParams())

waterfall = pd.DataFrame(
    {
        "trades": [len(r.trades) for r in steps.values()],
        "total_pct": [r.trades["net_pct"].sum() for r in steps.values()],
    },
    index=list(steps),
)
waterfall["delta_pct"] = waterfall["total_pct"].diff()
waterfall.round(2)


Os degraus 5 e 6 (comissão e swap) só existem em dinheiro, não em percentual de preço,
então entram depois da conversão — ver seção 7. Enquanto `COMMISSION_PER_LOT` e os swaps
estiverem zerados, **o número líquido abaixo ainda é otimista**.


In [ ]:
fig, ax = plt.subplots()
colors = ["#4c72b0" if v >= 0 else "#c44e52" for v in waterfall["total_pct"]]
ax.bar(range(len(waterfall)), waterfall["total_pct"], color=colors)
ax.axhline(0, color="black", linewidth=1)
ax.set_xticks(range(len(waterfall)))
ax.set_xticklabels(waterfall.index, rotation=20, ha="right")
ax.set_ylabel("Retorno somado (%)")
ax.set_title(f"O que sobra do resultado do notebook 01 - {ASSET} {TIMEFRAME}")
for i, v in enumerate(waterfall["total_pct"]):
    ax.text(i, v, f"{v:+.1f}%", ha="center", va="bottom" if v >= 0 else "top")
plt.tight_layout()
plt.show()


## 2. Esses ativos são reversíveis à média?

Uma pergunta que vem **antes** de otimizar parâmetro nenhum. O notebook 01 escolheu
ativos por retorno passado da estratégia — que é o mesmo dado que depois se usa para
julgá-la. Aqui a triagem usa propriedades do instrumento, independentes de qualquer
parâmetro:

- **Meia-vida OU** — em quantas barras um desvio da média se dissipa pela metade. Se ela
  for muito diferente de `InpExitBars`, a saída por tempo está desalinhada do próprio
  mecanismo que a estratégia diz explorar.
- **Hurst** — `< 0.5` reversão, `0.5` passeio aleatório, `> 0.5` tendência.
- **Variance ratio** no horizonte de saída — `< 1` reversão naquele horizonte
  específico, que é o que importa para quem segura por N barras.


In [ ]:
rows = []
for symbol in UNIVERSE:
    try:
        bars = data.load_rates(symbol, TIMEFRAME, START, END, allow_download=False)
    except (FileNotFoundError, ValueError) as exc:
        print(f"{symbol}: pulado ({exc.__class__.__name__})")
        continue
    rows.append(
        {
            "symbol": symbol,
            "bars": len(bars),
            "half_life_bars": indicators.ou_half_life(bars["close"]),
            "hurst": indicators.hurst_exponent(bars["close"]),
            f"var_ratio_{NB01_EXIT_BARS}": indicators.variance_ratio(bars["close"], NB01_EXIT_BARS),
        }
    )

reversion = pd.DataFrame(rows).set_index("symbol")
reversion.round(3)


**Como ler:** um ativo com Hurst acima de 0.5 e variance ratio acima de 1 não é candidato
a mean reversion nesse timeframe, por melhor que um par de parâmetros tenha parecido no
notebook 01. E se a meia-vida for de ~2 barras enquanto `InpExitBars=7`, a posição passa
a maior parte do tempo exposta *depois* que a reversão já aconteceu — o que a seção 4
mede diretamente.


## 3. Custo contra edge

O spread não é uma constante. Em AUDCHF M30 a mediana fica em torno de 2 pontos e a
média em 6 — a cauda (rollover, horários finos) domina. Como o edge por trade é da mesma
ordem de grandeza do spread, **em que hora se entra decide o sinal do resultado**.

O EA já tem `InpUseTradingHoursFilter`, então um achado aqui é acionável sem mexer no
MQL5.


In [ ]:
signals = run_backtest(df, ASSET, sp, er, CostModel(), ExecParams()).trades
signals["hour"] = pd.to_datetime(signals["entry_time"], utc=True).dt.hour
signals["spread_pct"] = signals["spread_cost_price"] / signals["entry_price"] * 100

by_hour = signals.groupby("hour").agg(
    trades=("net_pct", "size"),
    gross_pct=("gross_pct", "mean"),
    spread_pct=("spread_pct", "mean"),
    net_pct=("net_pct", "mean"),
    net_total=("net_pct", "sum"),
)
by_hour.round(5)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.bar(by_hour.index, by_hour["gross_pct"], label="edge bruto", color="#4c72b0")
ax1.bar(by_hour.index, -by_hour["spread_pct"], label="custo de spread", color="#c44e52")
ax1.axhline(0, color="black", linewidth=1)
ax1.set_xlabel("Hora de entrada (UTC)")
ax1.set_ylabel("Média por trade (%)")
ax1.set_title("Edge bruto contra custo, por hora")
ax1.legend()

colors = ["#55a868" if v >= 0 else "#c44e52" for v in by_hour["net_total"]]
ax2.bar(by_hour.index, by_hour["net_total"], color=colors)
ax2.axhline(0, color="black", linewidth=1)
ax2.set_xlabel("Hora de entrada (UTC)")
ax2.set_ylabel("Retorno líquido somado (%)")
ax2.set_title("Contribuição líquida acumulada por hora")

plt.tight_layout()
plt.show()

profitable = by_hour[by_hour["net_pct"] > 0]
print(f"Horas com média líquida positiva: {sorted(profitable.index.tolist())}")
print(f"Elas somam {profitable['net_total'].sum():+.2f}% em {int(profitable['trades'].sum())} trades,")
print(f"contra {by_hour['net_total'].sum():+.2f}% no total das {int(by_hour['trades'].sum())} entradas.")


> **Cuidado com esta célula.** Escolher as horas boas *olhando para o resultado* é
> exatamente o erro que a seção 5 existe para pegar. O que vale como achado é um padrão
> com explicação — spread alto no rollover, liquidez fina fora das sessões de Londres e
> Nova York — e não uma lista de horas que por acaso deram lucro. Qualquer filtro
> escolhido aqui precisa passar pelo walk-forward antes de virar `InpStartHour`.


## 4. A saída por X candles vale a pena?

Em duas etapas. Primeiro **MAE/MFE**: até onde o preço foi a favor e contra durante cada
trade. É a forma empírica de posicionar stop e alvo, em vez de chutar `2×ATR` e `2R`
como o EA faz hoje.

- Se a excursão **adversa dos vencedores** raramente passa de X, um stop em X quase não
  atrapalha e limita os perdedores.
- Se a excursão **favorável dos perdedores** chega rotineiramente a Y, esses trades
  estiveram no lucro em algum momento e um alvo em Y os teria capturado.


In [ ]:
baseline = run_backtest(df, ASSET, sp, er, CostModel(), ExecParams())
excursions = metrics.excursion_table(baseline.trades, POINT)
excursions.round(1)


In [ ]:
work = baseline.trades.assign(
    mae_points=baseline.trades["mae_price"].abs() / POINT,
    mfe_points=baseline.trades["mfe_price"] / POINT,
    outcome=np.where(baseline.trades["net_price"] > 0, "vencedor", "perdedor"),
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
for outcome, group in work.groupby("outcome"):
    ax1.hist(group["mae_points"], bins=60, alpha=0.55, label=outcome, density=True)
    ax2.hist(group["mfe_points"], bins=60, alpha=0.55, label=outcome, density=True)

ax1.set_title("Excursão adversa máxima (MAE)")
ax1.set_xlabel("pontos contra a posição")
ax2.set_title("Excursão favorável máxima (MFE)")
ax2.set_xlabel("pontos a favor da posição")
for ax in (ax1, ax2):
    ax.set_ylabel("densidade")
    ax.legend()
plt.tight_layout()
plt.show()


### Comparação das regras de saída

Todas rodam sobre **os mesmos sinais**, então a saída é a única coisa que varia. A coluna
`precisa_mudar_EA` marca o que hoje o `MeanRev1.mq5` não sabe fazer — um resultado bom
ali é uma proposta de mudança no MQL5, não algo que dá para ligar amanhã.

`ambiguo_%` é a fração de trades cuja barra de saída continha stop **e** alvo. Nessas o
motor assume stop primeiro; se essa fração for alta, a comparação das regras com ATR não
se sustenta sem dado intrabar.


In [ ]:
candidates = {
    "tempo (6)": ExitRules(time_bars=6),
    f"tempo ({NB01_EXIT_BARS}, notebook 01)": ExitRules(time_bars=NB01_EXIT_BARS),
    "tempo (12)": ExitRules(time_bars=12),
    "EA default: tempo 6 + SL 2xATR + TP 2R": exits.EA_DEFAULT,
    "SL 2xATR + TP 1R + teto 20": ExitRules(time_bars=20, atr_sl_mult=2.0, atr_tp_r=1.0),
    "trailing 2xATR + teto 20": ExitRules(time_bars=20, atr_sl_mult=2.0, atr_trailing_mult=2.0),
    "RSI volta a 50 + teto 20": ExitRules(time_bars=20, rsi_revert_level=50.0),
    "primeiro fecha no lucro + teto 20": ExitRules(time_bars=20, first_profitable_close=True),
    "toca a SMA curta + teto 20": ExitRules(time_bars=20, sma_touch=True),
}

rows = []
for label, rules in candidates.items():
    result = run_backtest(df, ASSET, sp, rules, CostModel(), ExecParams())
    stats = compute_stats(result.trades)
    rows.append(
        {
            "saida": label,
            "precisa_mudar_EA": rules.needs_ea_change,
            "trades": stats.trades,
            "net_%": round(stats.total_pct, 2),
            "media_%": round(stats.avg_pct, 5),
            "acerto": round(stats.win_rate, 3),
            "payoff": round(stats.payoff, 2),
            "profit_factor": round(stats.profit_factor, 2),
            "barras_med": round(stats.avg_bars_held, 1),
            "t_stat": round(stats.t_stat, 2),
            "ambiguo_%": round(result.ambiguous_fraction * 100, 1),
        }
    )

comparison = pd.DataFrame(rows).sort_values("net_%", ascending=False)
comparison


**A resposta à dúvida original** está na comparação entre as linhas de `tempo (N)` e as
demais. Três leituras possíveis, e cada uma leva a uma ação diferente:

- Se as saídas por tempo ficam no meio da tabela e uma regra de reversão do RSI domina, a
  saída fixa está cortando trades no meio do movimento — vale implementar no MQL5.
- Se todas as linhas são negativas, o problema não é a saída: é o custo, e nenhuma regra
  de saída resolve um edge menor que o spread.
- Se a variação entre elas for pequena comparada ao desvio-padrão dos retornos, a saída
  simplesmente não é o parâmetro que importa — e a busca deve ir para o filtro de entrada.


## 5. Superfície de parâmetros e walk-forward

A célula 34 do notebook 01 é uma lista de **picos**: o melhor `(streak, exit_bars)` de
cada ativo, escolhido olhando a amostra inteira. Esse procedimento produz uma lista assim
exista edge ou não.

Duas correções. Primeiro, julgar **planalto em vez de pico** — um ótimo cercado de
resultados ruins é ruído. Segundo, escolher em uma janela e medir em outra.


In [ ]:
STREAKS = [1, 2, 3, 4]
EXIT_BARS = [2, 3, 4, 5, 6, 7, 8, 10, 12]

budget = validation.SearchBudget()
surface = validation.parameter_surface(
    df, ASSET, STREAKS, EXIT_BARS, costs=CostModel(), exec_params=ExecParams(), budget=budget
)
plateau = validation.plateau_score(surface)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 4.5))
limit = np.nanmax(np.abs(surface.to_numpy()))
sns.heatmap(surface, annot=True, fmt=".1f", center=0, vmin=-limit, vmax=limit, cmap="RdYlGn", ax=ax1)
ax1.set_title("Retorno líquido somado (%) - superfície crua")
sns.heatmap(plateau, annot=True, fmt=".1f", center=0, vmin=-limit, vmax=limit, cmap="RdYlGn", ax=ax2)
ax2.set_title("Média com os vizinhos - escolher o melhor DESTE")
plt.tight_layout()
plt.show()

print(budget)


In [ ]:
folds, oos = validation.walk_forward(
    df, ASSET, STREAKS, EXIT_BARS,
    train_months=24, test_months=6,
    costs=CostModel(), exec_params=ExecParams(), budget=budget,
)
folds


A instabilidade das colunas `chosen_streak` / `chosen_exit_bars` é, por si só, um
resultado: se cada janela de treino escolhe um parâmetro diferente, não existe parâmetro
a escolher — existe ruído sendo ajustado.


In [ ]:
if oos.empty:
    print("Sem janelas out-of-sample suficientes para o tamanho de treino/teste pedido.")
else:
    oos_sorted = oos.sort_values("exit_time").reset_index(drop=True)
    equity = oos_sorted["net_pct"].cumsum()

    # Benchmark honesto: buy-and-hold de verdade no mesmo período, não o cumsum de
    # janelas sobrepostas que o notebook 01 chamava de "asset_variation".
    test_start = oos_sorted["entry_time"].min()
    held = df[df["time"] >= test_start]
    buy_hold = (held["close"] / held["close"].iloc[0] - 1) * 100

    fig, ax = plt.subplots()
    ax.plot(oos_sorted["exit_time"], equity, label="estratégia, líquido, out-of-sample")
    ax.plot(held["time"], buy_hold, label="buy & hold", alpha=0.6)
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.set_ylabel("Retorno acumulado (%)")
    ax.set_title(f"Out-of-sample concatenado - {ASSET} {TIMEFRAME}")
    ax.legend()
    plt.tight_layout()
    plt.show()

    in_sample = compute_stats(run_backtest(df, ASSET, sp, er, CostModel(), ExecParams()).trades)
    print(f"In-sample (parâmetros do notebook 01): {in_sample.total_pct:+.2f}% em {in_sample.trades} trades")
    print(f"Out-of-sample (walk-forward)         : {oos['net_pct'].sum():+.2f}% em {len(oos)} trades")


## 6. O resultado se distingue de ruído?

Três coisas diferentes, que o notebook 01 tratava como uma só.

**`signal_to_noise`** é o `n_std` do 01: `média / desvio`. Mede a qualidade de um trade
individual. Um valor de 0.04 quer dizer que o retorno típico é uma fração minúscula da
variação típica — o que é normal e não condena a estratégia.

**`t_stat`** é `média / desvio × √n`. É o teste de a média diferir de zero. Com milhares
de trades ele fica grande mesmo com edge ínfimo, e é por isso que **significância
estatística não é edge econômico**: o mesmo conjunto que dá t≈3.6 bruto fica negativo
depois do spread.

**Teste de permutação** é o mais duro dos três: roda a estratégia em séries com a mesma
distribuição de tamanho de barra e o mesmo calendário, mas sem estrutura serial. Se o
resultado real não se destaca dessa distribuição, não há o que explorar.

O teste é feito **em bruto e em líquido, sobre as mesmas séries embaralhadas**. Os dois
juntos separam duas conclusões que um número só confunde:

| bruto | líquido | leitura |
|---|---|---|
| dentro da nula | dentro da nula | não há estrutura; a ideia não se sustenta |
| **fora** da nula | dentro da nula | **a estrutura existe, mas é menor que o custo** |
| fora da nula | fora da nula | edge que sobrevive ao custo |

A linha do meio é a que muda o rumo da pesquisa: significa atacar o custo (filtro de
horário, timeframe maior, alvo mais largo para amortizar o spread) em vez de abandonar o
sinal.

> Esta célula roda `n_permutations` backtests completos (~0,7 s cada). Com 200, conte
> uns 3 minutos.


In [ ]:
final = run_backtest(df, ASSET, sp, er, CostModel(), ExecParams())
stats = compute_stats(final.trades)

pd.Series(
    {
        "trades": stats.trades,
        "total liquido (%)": round(stats.total_pct, 2),
        "media por trade (%)": round(stats.avg_pct, 5),
        "signal_to_noise (n_std do nb01)": round(stats.signal_to_noise, 4),
        "t_stat": round(stats.t_stat, 2),
        "acerto": round(stats.win_rate, 3),
        "profit factor": round(stats.profit_factor, 3),
    },
    name=f"{ASSET} {TIMEFRAME}",
).to_frame()


In [ ]:
# Reduza n_permutations se estiver lento; 200 dá resolução de p ate ~0.005.
perm = validation.permutation_test(
    df, ASSET, sp, er, costs=CostModel(), exec_params=ExecParams(),
    n_permutations=200, seed=20260924,
)

titles = {"gross_pct": "bruto (antes do spread)", "net_pct": "líquido (depois do spread)"}
fig, axes = plt.subplots(1, len(perm["columns"]), figsize=(16, 5), sharey=True)

for ax, column in zip(np.atleast_1d(axes), perm["columns"]):
    stats = perm["by_column"][column]
    ax.hist(stats["null"], bins=40, alpha=0.75, label="séries embaralhadas")
    ax.axvline(stats["actual"], color="#c44e52", linewidth=2,
               label=f"real = {stats['actual']:+.1f}%")
    ax.set_title(f"{titles.get(column, column)}  (z = {stats['z']:+.2f})")
    ax.set_xlabel("Retorno somado (%)")
    ax.legend()
np.atleast_1d(axes)[0].set_ylabel("frequência")
fig.suptitle(f"Distribuição nula - {perm['n_permutations']} permutações - {ASSET} {TIMEFRAME}")
plt.tight_layout()
plt.show()

display(validation.permutation_summary(perm))

print(f"busca: {budget}")
print()
print("O p-value NÃO está corrigido para o número de combinações testadas acima.")
print("Com essa contagem de buscas, um p de 0.01 isolado não é surpreendente.")


## 7. Exportar para o `risk-mgmt`

O ledger sai no formato `<symbol>_trades.csv` que o `risk_mgmt/logsource/idxswing91_csv.py`
já lê. Nenhum dos dois projetos importa o outro — o contrato é o arquivo, como pede o
`CLAUDE.md` do Lanius.

Para converter preço em dinheiro é preciso o `value_per_point` do símbolo
(`tick_value / tick_size`). Ele **não** está no CSV de rates: vem do MT5 aberto ou do
`risk-mgmt/config/symbols.yaml`. Zero é rejeitado de propósito — foi o bug do ASXAUD em
19/09, em que todo o P/L virou `0.00` sem nenhum erro.


In [ ]:
# PREENCHER com o valor real do símbolo (MT5: clique direito -> Especificação).
VALUE_PER_POINT = None  # ex.: para um par com tick_value=1.0 e tick_size=0.00001 -> 100_000

if VALUE_PER_POINT is None:
    print("VALUE_PER_POINT não preenchido - conversão em dinheiro e export pulados.")
    print("Até lá, as seções acima seguem válidas: elas trabalham em % de preço,")
    print("que não depende do spec do símbolo.")
else:
    priced = to_money(
        final.trades,
        value_per_point=VALUE_PER_POINT,
        point=POINT,
        sizing=Sizing(mode="fixed", lots=0.10),   # lote fixo: cenário 3 do risk-mgmt fica exato
        commission_per_lot=COMMISSION_PER_LOT,
        swap_long_per_lot=SWAP_LONG_PER_LOT,
        swap_short_per_lot=SWAP_SHORT_PER_LOT,
    )
    money_stats = compute_stats(priced, initial_balance=10_000.0)

    print(f"P/L bruto      : {priced['pnl_gross'].sum():+.2f}")
    print(f"  spread       : {priced['pnl_spread'].sum():+.2f}")
    print(f"  comissão     : {-priced['commission'].sum():+.2f}")
    print(f"  swap         : {priced['swap'].sum():+.2f}")
    print(f"P/L líquido    : {money_stats.total_money:+.2f}")
    print(f"Drawdown máx.  : {money_stats.max_drawdown_money:.2f} ({money_stats.max_drawdown_pct:.2f}%)")

    out = export.write_trades_csv(priced, Path("..") / ".." / "risk-mgmt" / "reports" / "meanrev")
    print(f"\nEscrito: {out}")
    print("Depois, no risk-mgmt (venv PRÓPRIO dele):")
    print("  python scripts/run_risk_report.py --mode local --strategy MeanRev1=reports/meanrev --dry-run")


---

## O que este notebook não resolve

- **Ordem intrabar.** Quando uma barra contém stop e alvo, o motor assume stop. A coluna
  `ambiguo_%` da seção 4 diz quanto do resultado depende disso. Resolver exige carregar
  M1 e reconstruir o caminho dentro da barra.
- **Comissão e swap** continuam sendo parâmetros até que os valores reais da corretora
  sejam preenchidos. Enquanto estiverem em zero, todo número líquido aqui é otimista.
- **Reconciliação externa.** Paridade com o notebook 01 prova consistência interna, não
  correspondência com a execução. A prova é rodar o `MeanRev1` no Strategy Tester com os
  mesmos parâmetros e comparar contagem de trades e P/L. Diferença no preenchimento tick
  a tick é esperada; diferença na contagem de trades é bug.
- **Uma estratégia, um ativo por vez.** Carteira e saldo compartilhado são o trabalho do
  `risk-mgmt`, alimentado pela seção 7.
